# Churn — nettoyage des tables et construction de la table finale

**Entrée** : 6 fichiers CSV dans `data/raw/`
**Sortie** : `data/final/all_tables.csv` — 34 000 lignes × 10 colonnes

| Fichier source | Contenu | Grain |
|---|---|---|
| `CIBLE_CHURN_SAMPLE2_new.csv` | profil du client + label | 1 ligne / contrat |
| `DATA_...csv` | consommation internet | 1 ligne / contrat / mois |
| `ENTRANT_...csv` | appels et SMS **reçus** | 1 ligne / contrat / mois |
| `SORTANT_...csv` | appels et SMS **émis** | 1 ligne / contrat / mois |
| `RECHARGE_...csv` | recharges | 1 ligne / contrat / mois |
| `USSD_...csv` | achats de forfaits | 1 ligne / contrat / mois |

## 1. Préparation

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# dossiers du projet
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
FINAL = ROOT / 'data' / 'final'
for d in (PROCESSED, FINAL):
    d.mkdir(parents=True, exist_ok=True)

FICHIERS = {
    'cible':    'CIBLE_CHURN_SAMPLE2_new.csv',
    'data':     'DATA_CHURN_2NW_2025.csv',
    'entrant':  'ENTRANT_CHURN_2NW_2025.csv',
    'sortant':  'SORTANT_CHURN_2NW_2025.csv',
    'recharge': 'RECHARGE_CHURN_2NW_2025.csv',
    'ussd':     'USSD_CHURN_2NW_2025.csv',
}

def charger(nom):
    """Lit un fichier brut et enlève les espaces autour des noms et des valeurs."""
    df = pd.read_csv(RAW / FICHIERS[nom], sep=';', dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].str.strip()
    return df

print('Racine du projet :', ROOT)
print('Fichiers déclarés :', len(FICHIERS))

Racine du projet : c:\Users\hayet\Downloads\ML churn\ML
Fichiers déclarés : 6


## 2. Nettoyage, table par table

### 2.1 Table `cible`

**Règles de calcul du label** :

- `churn = 1` si le client part (`D`) ou est suspendu (`S`)
- `churn = 0` s'il reste actif (`A`)

In [8]:
# --- nettoyage de la table cible ---
cible = charger('cible')

# 1. le label, calculé AVANT de perdre statut_next_month
cible['churn'] = (cible['statut_next_month'] != 'A').astype(int)

# 2. on garde seulement les colonnes utiles, suppression de ANC_J
COLONNES_CIBLE = ['CODE_CONTRAT', 'ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn']
cible = cible[COLONNES_CIBLE].copy()

# 3. ANC_M en nombre
cible['ANC_M'] = pd.to_numeric(cible['ANC_M'])

print('cible nettoyée :', cible.shape)
print('colonnes       :', list(cible.columns))
print('taux de churn  :', round(cible['churn'].mean() * 100, 2), '%')
print()
print(cible.head(3).to_string(index=False))

cible nettoyée : (34000, 6)
colonnes       : ['CODE_CONTRAT', 'ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn']
taux de churn  : 28.12 %

CODE_CONTRAT  ANC_M HANDSET          OFFRE STATUT  churn
 SN_10004606    177      4G PRE - TT 1500%      A      0
 SN_10006819    177      2G   PRE - Double      A      0
 SN_10025174    178      4G PRE - TT 1500%      A      0


---

### 2.2 Les 5 tables mensuelles

In [10]:
# colonnes à supprimer, table par table
A_SUPPRIMER = {
    'recharge': ['STATUT'],                      # constant sur toute l'année
    'sortant':  ['Duree_offnet_tot',             # = somme des 4 opérateurs
                 'revenu_inter', 'revenu_roaming',
                 'revenu_inter_roaming',
                 'DUREE_APPEL_LYCA_TOT', 'DUREE_APPEL_NESSMA_TOT'],
    'entrant':  ['DUREE_OFFNET_IN', 'NB_SMS_OFFNET_IN',   # = Ooredoo + Orange
                 'DUREE_TT_FIXE_IN', 'DUREE_APPEL_INTER_IN'],
    'data':     [],
    'ussd':     [],
}

# colonnes gardées après agrégation
MESURES = {
    'data':     ['VOLUME_SESSION'],
    'entrant':  ['DUREE_APPEL_IN', 'NB_SMS_IN'],
    'sortant':  ['DUREE_APPEL_TOT', 'nb_sms_tot'],
    'recharge': ['MNT_RECH'],
    'ussd':     ['MNT_FORFAIT'],
}
print('configuration prête')

configuration prête


### 2.3 La fonction de nettoyage commune

In [11]:
def nettoyer_mensuelle(nom, dernier_mois=11):
    """Nettoie une table mensuelle (1 ligne par contrat et par mois).

    - retire les colonnes inutiles
    - retire les lignes sans date (client inactif toute l'année)
    - convertit les colonnes de chiffres
    """
    df = charger(nom)
    avant = len(df)

    # étape 1 : colonnes inutiles
    df = df.drop(columns=[c for c in A_SUPPRIMER.get(nom, []) if c in df.columns])

    # étape 2 : lignes sans date
    df['mois'] = pd.to_datetime(df['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
    df = df.dropna(subset=['mois'])

    # étape 3 : fenêtre d'observation
    df = df[df['mois'].dt.month <= dernier_mois].copy()

    # étape 4 : typage numérique
    for c in df.columns:
        if c in ('CODE_CONTRAT', 'MONTH_DT', 'mois'):
            continue
        df[c] = pd.to_numeric(df[c], errors='coerce')

    print(f'{nom:9s} {avant:>7,} -> {len(df):>7,} lignes  ({avant - len(df):>6,} supprimées)')
    return df

tables = {}
for nom in ['data', 'entrant', 'sortant', 'recharge', 'ussd']:
    tables[nom] = nettoyer_mensuelle(nom)

data      164,563 -> 143,210 lignes  (21,353 supprimées)
entrant   352,982 -> 322,965 lignes  (30,017 supprimées)
sortant   330,000 -> 303,468 lignes  (26,532 supprimées)
recharge  282,241 -> 259,682 lignes  (22,559 supprimées)
ussd      145,793 -> 125,110 lignes  (20,683 supprimées)


In [12]:
# contrôle : plus aucune date vide, décembre a disparu
for nom, df in tables.items():
    m_min = int(df['mois'].dt.month.min())
    m_max = int(df['mois'].dt.month.max())
    print(f'{nom:9s} mois de {m_min} à {m_max} | dates vides : {df["mois"].isna().sum()}')

data      mois de 1 à 11 | dates vides : 0
entrant   mois de 1 à 11 | dates vides : 0
sortant   mois de 1 à 11 | dates vides : 0
recharge  mois de 1 à 11 | dates vides : 0
ussd      mois de 1 à 11 | dates vides : 0


## 3. Agrégation : de 12 lignes à 1 ligne par contrat

In [13]:
# --- moyennes mensuelles (niveau d'usage) ---
blocs = {}
for nom in ['data', 'recharge', 'ussd']:
    blocs[nom] = tables[nom].groupby('CODE_CONTRAT')[MESURES[nom]].mean()
    print(f'{nom:9s} -> {len(blocs[nom]):>6,} contrats | moyenne de {list(blocs[nom].columns)}')

# --- sommes pour les appels et SMS (volume cumulé) ---
recu = tables['entrant'][['CODE_CONTRAT', 'mois', 'DUREE_APPEL_IN', 'NB_SMS_IN']]
emis = tables['sortant'][['CODE_CONTRAT', 'mois', 'DUREE_APPEL_TOT', 'nb_sms_tot']]

mensuel = recu.merge(emis, on=['CODE_CONTRAT', 'mois'], how='outer')
mensuel['APPEL_TOTAL'] = mensuel['DUREE_APPEL_IN'].fillna(0) + mensuel['DUREE_APPEL_TOT'].fillna(0)
mensuel['SMS_TOTAL'] = mensuel['NB_SMS_IN'].fillna(0) + mensuel['nb_sms_tot'].fillna(0)

blocs['appels'] = mensuel.groupby('CODE_CONTRAT')[['APPEL_TOTAL', 'SMS_TOTAL']].sum()
print(f'{"appels":9s} -> {len(blocs["appels"]):>6,} contrats | somme de {list(blocs["appels"].columns)}')

# --- la cible est déjà à 1 ligne par contrat ---
blocs['cible'] = cible.set_index('CODE_CONTRAT')
print(f'{"cible":9s} -> {len(blocs["cible"]):>6,} contrats | {list(blocs["cible"].columns)}')

data      -> 24,271 contrats | moyenne de ['VOLUME_SESSION']
recharge  -> 33,783 contrats | moyenne de ['MNT_RECH']
ussd      -> 23,372 contrats | moyenne de ['MNT_FORFAIT']
appels    -> 33,972 contrats | somme de ['APPEL_TOTAL', 'SMS_TOTAL']
cible     -> 34,000 contrats | ['ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn']


### Vérification sur un contrat réel

On prend `SN_10004606` et on recalcule sa moyenne data **et** sa somme d'appels à la main.

In [14]:
sn = 'SN_10004606'

# moyenne data
b = charger('data')
b = b[b['CODE_CONTRAT'] == sn].copy()
b['mois'] = pd.to_datetime(b['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
b = b.dropna(subset=['mois'])
b = b[b['mois'].dt.month <= 11]
b['VOLUME_SESSION'] = pd.to_numeric(b['VOLUME_SESSION'])

print(f'--- moyenne data ({len(b)} mois) ---')
print('recalculée ici :', round(b['VOLUME_SESSION'].mean(), 2))
print('dans le bloc   :', round(blocs['data'].loc[sn, 'VOLUME_SESSION'], 2))

# somme appels
e = charger('entrant')
e = e[e['CODE_CONTRAT'] == sn].copy()
e['mois'] = pd.to_datetime(e['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
e = e.dropna(subset=['mois']); e = e[e['mois'].dt.month <= 11]
e['DUREE_APPEL_IN'] = pd.to_numeric(e['DUREE_APPEL_IN'])

s = charger('sortant')
s = s[s['CODE_CONTRAT'] == sn].copy()
s['mois'] = pd.to_datetime(s['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
s = s.dropna(subset=['mois']); s = s[s['mois'].dt.month <= 11]
s['DUREE_APPEL_TOT'] = pd.to_numeric(s['DUREE_APPEL_TOT'])

total = (e['DUREE_APPEL_IN'].sum() + s['DUREE_APPEL_TOT'].sum())
print(f'--- somme appels ---')
print('recalculée ici :', round(total, 2))
print('dans le bloc   :', round(blocs['appels'].loc[sn, 'APPEL_TOTAL'], 2))

--- moyenne data (11 mois) ---
recalculée ici : 5277.1
dans le bloc   : 5277.1
--- somme appels ---
recalculée ici : 7359.43
dans le bloc   : 7359.43


In [15]:
# sauvegarde des blocs intermédiaires
for nom, bloc in blocs.items():
    bloc.reset_index().to_csv(PROCESSED / f'agg_{nom}.csv', index=False)

print('fichiers écrits dans', PROCESSED)
for f in sorted(PROCESSED.glob('agg_*.csv')):
    d = pd.read_csv(f, nrows=1, keep_default_na=False)
    print(f'  {f.name:20s} {len(d.columns)} colonnes')

fichiers écrits dans c:\Users\hayet\Downloads\ML churn\ML\data\processed
  agg_appels.csv       3 colonnes
  agg_cible.csv        6 colonnes
  agg_data.csv         2 colonnes
  agg_entrant.csv      3 colonnes
  agg_recharge.csv     2 colonnes
  agg_sortant.csv      3 colonnes
  agg_ussd.csv         2 colonnes


## 4. Réunir les tables en une seule

### Le principe

```
        cible  (34 000 contrats)   ← la base, elle porte le label
          │
          │  left join sur CODE_CONTRAT
          ├────────┬────────┬────────┐
          ▼        ▼        ▼        ▼
        data  recharge  ussd    appels
```

In [ ]:
# on part de la cible
table_finale = blocs['cible'].copy()
print('départ :', table_finale.shape)

# on ajoute les blocs un par un
for nom in ['data', 'recharge', 'ussd', 'appels']:
    table_finale = table_finale.join(blocs[nom], how='left')
    print(f'+ {nom:9s} -> {table_finale.shape[0]:,} lignes x {table_finale.shape[1]} colonnes')

print()
print('forme finale :', table_finale.shape)

# le contrôle se fait AVANT de retirer la clé
assert len(table_finale) == 34000, 'la jointure a dupliqué des lignes !'
print('contrôle : 34 000 lignes, 1 par contrat ✓')

départ : (34000, 5)
+ data      -> 34,000 lignes x 6 colonnes
+ recharge  -> 34,000 lignes x 7 colonnes
+ ussd      -> 34,000 lignes x 8 colonnes
+ appels    -> 34,000 lignes x 10 colonnes

forme finale : (34000, 10)
contrôle : 34 000 lignes, 1 par contrat ✓


## 5. Export de la table finale

In [17]:
sortie = table_finale.reset_index().drop(columns=['CODE_CONTRAT'])
sortie.to_csv(FINAL / 'all_tables.csv', index=False)

print('fichier écrit :', FINAL / 'all_tables.csv')
print('forme         :', sortie.shape)
print('colonnes      :', list(sortie.columns))
print()
print(sortie.head(3).to_string(index=False))

fichier écrit : c:\Users\hayet\Downloads\ML churn\ML\data\final\all_tables.csv
forme         : (34000, 10)
colonnes      : ['ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn', 'VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT', 'APPEL_TOTAL', 'SMS_TOTAL']

 ANC_M HANDSET          OFFRE STATUT  churn  VOLUME_SESSION  MNT_RECH  MNT_FORFAIT  APPEL_TOTAL  SMS_TOTAL
   177      4G PRE - TT 1500%      A      0     5277.095455 21.727273    14.281818      7359.43     1768.0
   177      2G   PRE - Double      A      0             NaN  5.222222     0.800000       598.95      592.0
   178      4G PRE - TT 1500%      A      0     5242.772727 27.727273     1.737500     10223.95     2290.0
